## Langfuse Claude run: score counts vs sanitized `custom_id`

From `ingest.py`:
- **`accuracy`** / **`error_type`**: only if `custom_id` is in the ground-truth map (`expected is not None`).
- **`cost_usd`**: whenever pricing exists for the model (`cost` is not None).

Langfuse **redacts** some `metadata.custom_id` values into `-hash_<b64>--`. Ingest still sends the real id, but the **API returns** the sanitized string. Those ids no longer match `ground_truth` keys → **no accuracy/error_type**, but the trace still gets **cost_usd**.

On exported CSVs (`streamlit_app/langfuse_*.csv`), GHA runs like `gha-23832575964` show **300** `cost_usd` vs **267** `accuracy`/`error_type` per Claude model — gap **33** = rows whose stored `metadata.custom_id` is `-hash_...--` (all **33** missing-accuracy traces are sanitized in the haiku run checked below).

In [1]:
import csv
import re
from collections import Counter, defaultdict

BASE = "../streamlit_app"  # from notebooks/
SCORES = f"{BASE}/langfuse_scores.csv"
TRACES = f"{BASE}/langfuse_traces.csv"

SANITIZED = re.compile(r"^-hash_.+--$")


def extract_run(tags_s: str | None) -> str | None:
    if not tags_s:
        return None
    m = re.search(r"run:([^,\]]+)", str(tags_s))
    return m.group(1).strip(" '\"") if m else None


def extract_model(tags_s: str | None) -> str | None:
    if not tags_s:
        return None
    m = re.search(r"model:([^,\]]+)", str(tags_s))
    return m.group(1).strip(" '\"") if m else None


def scores_for(run_id: str, model: str) -> list[dict]:
    out = []
    with open(SCORES, newline="", encoding="utf-8") as f:
        for row in csv.DictReader(f):
            tags = str(row.get("trace.tags") or "")
            if run_id in tags and model in tags:
                out.append(row)
    return out


def traces_for(run_id: str, model: str) -> list[dict]:
    out = []
    with open(TRACES, newline="", encoding="utf-8") as f:
        for row in csv.DictReader(f):
            tags = str(row.get("tags") or "")
            if run_id in tags and model in tags:
                out.append(row)
    return out


def compare_run(run_id: str, model: str) -> dict:
    sc = scores_for(run_id, model)
    tr = traces_for(run_id, model)
    names = Counter(r["name"] for r in sc)
    by_trace: dict[str, set[str]] = defaultdict(set)
    for r in sc:
        by_trace[r["traceId"]].add(r["name"])
    cost_no_acc = [tid for tid, n in by_trace.items() if "cost_usd" in n and "accuracy" not in n]
    tid_to_cid = {t["id"]: t.get("metadata.custom_id") for t in tr}
    sanitized_tr = [t for t in tr if SANITIZED.match(str(t.get("metadata.custom_id") or ""))]
    return {
        "score_counts": dict(names),
        "n_traces": len(tr),
        "n_cost_without_accuracy": len(cost_no_acc),
        "n_sanitized_custom_id": len(sanitized_tr),
        "cost_no_acc_all_sanitized": all(
            SANITIZED.match(str(tid_to_cid.get(tid) or "")) for tid in cost_no_acc
        ),
        "sample_sanitized_cids": [tid_to_cid[tid] for tid in cost_no_acc[:5]],
    }


# Example: GHA batch run (adjust RUN / MODEL to match what you see in Langfuse)
RUN = "gha-23832575964"
MODEL = "claude-haiku-4-5"
compare_run(RUN, MODEL)

{'score_counts': {'error_type': 267, 'accuracy': 267, 'cost_usd': 300},
 'n_traces': 300,
 'n_cost_without_accuracy': 33,
 'n_sanitized_custom_id': 33,
 'cost_no_acc_all_sanitized': True,
 'sample_sanitized_cids': ['-hash_bFXJEg8-dwR1kGZVEGWYDA--',
  '-hash_1iiULM6KMdhzPNxFHXahug--',
  '-hash_HKXcWjhnqAMHuAm-fECS3w--',
  '-hash_p5E3G1l5twN516qzgeB77A--',
  '-hash_AZnzm5yY-WXRhjDMETaXHg--']}

In [3]:
# Optional: list all Claude runs where cost_usd count != accuracy count
from collections import defaultdict

by_key: dict[tuple[str, str], list[dict]] = defaultdict(list)
with open(SCORES, newline="", encoding="utf-8") as f:
    for row in csv.DictReader(f):
        tags = str(row.get("trace.tags") or "")
        if "claude" not in tags.lower():
            continue
        m, r = extract_model(tags), extract_run(tags)
        if m and r:
            by_key[(m, r)].append(row)

mismatches = []
for (m, r), rows in sorted(by_key.items()):
    c = Counter(x["name"] for x in rows)
    a, cost = c.get("accuracy", 0), c.get("cost_usd", 0)
    if a != cost:
        mismatches.append((m, r, dict(c)))

mismatches, f"... total {len(mismatches)} (model, run) pairs with accuracy != cost_usd"

([('claude-haiku-4-5',
   'gha-23349348405',
   {'accuracy': 267, 'cost_usd': 300, 'error_type': 267}),
  ('claude-haiku-4-5',
   'gha-23832575964',
   {'error_type': 267, 'accuracy': 267, 'cost_usd': 300}),
  ('claude-sonnet-4-5',
   'gha-23349348405',
   {'cost_usd': 300, 'accuracy': 267, 'error_type': 267}),
  ('claude-sonnet-4-5',
   'gha-23832575964',
   {'cost_usd': 300, 'error_type': 267, 'accuracy': 267}),
  ('claude-sonnet-4-6',
   'gha-23349348405',
   {'cost_usd': 300, 'accuracy': 267, 'error_type': 267}),
  ('claude-sonnet-4-6',
   'gha-23832575964',
   {'cost_usd': 300, 'accuracy': 267, 'error_type': 267})],
 '... total 6 (model, run) pairs with accuracy != cost_usd')